# tDCBAM - CEDAR Dataset Evaluation



## Step 1 — Imports & Reproducibility

In [ ]:
import os, sys, json, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = os.path.abspath(os.path.join(os.path.abspath(os.getcwd()), '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from models.feature_extractor import DenseNetFeatureExtractor
from losses.triplet_loss       import TripletLoss
from utils.model_evaluation    import (compute_metrics, _plot_det_curve,
                                        _plot_far_frr, _plot_confusion_matrix,
                                        _plot_score_distribution, _plot_roc_curve)
from dataloader.tDCBAM_trainloader import (get_transforms,
                                            preprocess_image,
                                            sample_augment_params)

def seed_everything(seed=42):
    """Seed all random sources for full reproducibility."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f" > [Seed] {seed}")

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" > [Device] {DEVICE}" +
      (f"  ({torch.cuda.get_device_name()})" if torch.cuda.is_available() else ""))


/home/lawrence/workspace/thesis/thesis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


 > [Seed] 42
 > [Device] cuda  (NVIDIA GeForce RTX 5080)


## Step 2 — Configuration

In [ ]:
NOTEBOOK_NAME = 'triplet_cedar'
DATASET       = 'cedar'
DATASET_NAME  = 'CEDAR'

SPLIT_DIR      = os.path.join(REPO_ROOT, 'data',        'ratio_splits')
CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints', 'ablation_splits')
EVAL_DIR       = os.path.join(REPO_ROOT, 'model_evals',  NOTEBOOK_NAME)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EVAL_DIR,       exist_ok=True)

SPLIT_RATIOS = ['70_15_15']

IMG_SIZE    = 224
INPUT_SHAPE = (IMG_SIZE, IMG_SIZE)
NUM_WORKERS = 4

# ── Hyperparameters — identical to proposed model training ────────────────────
# Training paradigm is metric learning, identical to proposed.
# Only architectural difference: no CBAM (baseline=True).
TRAIN_EPOCHS        = 100
TRAIN_PHASE1_EPOCHS = 20
TRAIN_LR            = 1e-4
TRAIN_MARGIN        = 1.0
TRAIN_WEIGHT_DECAY  = 1e-4
TRAIN_BATCH_SIZE    = 32

FEATURE_DIM = 1024

print(f" > [Ablation B] DenseNet-121 + Triplet Network — No CBAM")
print(f" > [Config] Epochs: {TRAIN_EPOCHS} (P1 frozen: {TRAIN_PHASE1_EPOCHS})")
print(f" > [Config] LR: {TRAIN_LR} | Margin: {TRAIN_MARGIN} | "
      f"WD: {TRAIN_WEIGHT_DECAY} | Batch: {TRAIN_BATCH_SIZE}")
print(f" > [Config] CBAM: OFF | L2 Norm: ON | Loss: TripletLoss (SED)")

 > [Config] Epochs: 100 (P1 frozen: 20)
 > [Config] LR: 0.0001 | Margin: 1.0 | WD: 0.0001 | Batch: 32


## Step 3 — Transforms

In [ ]:
train_transform = get_transforms(mode='train', input_shape=INPUT_SHAPE)
val_transform   = get_transforms(mode='val',   input_shape=INPUT_SHAPE)

print(" > [Transforms] train_transform: augmentation ON  (independent per image)")
print(" > [Transforms] val_transform  : augmentation OFF (preprocessing only)")

 > [Transforms] train_transform: augmentation ON  (geometric)
 > [Transforms] val_transform  : augmentation OFF (preprocessing only)


## Step 4 — Datasets

In [ ]:
class SplitTripletDataset(Dataset):
    """
    Triplet dataset for triplet-loss training.

    Generates (Anchor, Positive, Negative) triplets with offline hard
    negative mining:
        hard_neg_ratio  of negatives are skilled forgeries from the
                        same writer (hard negatives).
        1-hard_neg_ratio of negatives are genuine signatures from a
                        randomly chosen other writer (easy negatives).

    Triplets are regenerated each epoch via _generate_triplets() to prevent
    the model from memorising fixed pairings across epochs.

    Augmentation strategy (training=True):
        Flip    : Sampled ONCE per triplet, shared across anchor, positive,
                  and negative. Ensures orientation is consistent within
                  a triplet — inconsistent flip would corrupt the loss signal
                  by introducing directional mismatch as a training artifact.

        Rotation, zoom, jitter : Sampled INDEPENDENTLY per image. Forces the
                  model to learn stroke-level similarity invariant to scale,
                  rotation, and position variation. This is the primary
                  mechanism for learning spatial invariance.

    Augmentation strategy (training=False):
        val_transform is applied per-image (no augmentation).

    Args:
        user_dict      (dict)    : {uid: {'genuine': [...], 'forged': [...]}}
        input_shape    (tuple)   : Canvas size. Default (224, 224).
        val_transform  (callable): Transform for val/test. No augmentation.
        training       (bool)    : True for train split, False for val/test.
        hard_neg_ratio (float)   : Fraction of negatives that are skilled
                                   forgeries. Default 0.7.
    """

    def __init__(self, user_dict, input_shape=(224, 224),
                 val_transform=None, training=True, hard_neg_ratio=0.7):
        self.input_shape    = input_shape
        self.val_transform  = val_transform
        self.training       = training
        self.hard_neg_ratio = hard_neg_ratio

        self.user_genuine_map  = {}
        self.user_forged_map   = {}
        self.all_genuine_paths = []

        for uid, data in user_dict.items():
            gen_key  = next((k for k in data if k.lower() in
                             ('genuine', 'gen')), None)
            forg_key = next((k for k in data if k.lower() in
                             ('forged', 'forgeries', 'forg')), None)
            gen_paths  = data.get(gen_key,  []) if gen_key  else []
            forg_paths = data.get(forg_key, []) if forg_key else []
            if len(gen_paths) >= 2:
                self.user_genuine_map[uid] = gen_paths
                self.user_forged_map[uid]  = forg_paths
                self.all_genuine_paths.extend((p, uid) for p in gen_paths)

        self.users = list(self.user_genuine_map.keys())
        self._generate_triplets()
        mode_label = "triplet-level aug" if training else "no aug"
        print(f"   TripletDataset: {len(self.triplets)} triplets | "
              f"{len(self.users)} users | {mode_label}")

    def _generate_triplets(self):
        """
        Regenerate all triplets. Call at the end of each epoch.

        Hard negative selection:
            With probability hard_neg_ratio: skilled forgery of same writer.
            Otherwise: genuine from a randomly chosen other writer.
        """
        self.triplets = []
        for anchor_path, uid in self.all_genuine_paths:
            positives = [
                p for p in self.user_genuine_map[uid] if p != anchor_path
            ]
            if not positives:
                continue
            pos_path  = random.choice(positives)
            forgeries = self.user_forged_map.get(uid, [])

            if random.random() < self.hard_neg_ratio and forgeries:
                neg_path = random.choice(forgeries)
            else:
                other_uid = random.choice(
                    [u for u in self.users if u != uid]
                )
                neg_path = random.choice(self.user_genuine_map[other_uid])

            self.triplets.append((anchor_path, pos_path, neg_path))

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a_path, p_path, n_path = self.triplets[idx]

        if self.training:
            # ── Sample flip once — shared across all three images ─────────────
            # Anchor, positive, and negative must share the same flip decision.
            # If flip were sampled independently, the model would see
            # orientation-mismatched triplets and learn directional artifacts
            # instead of writer identity features.
            #
            # Rotation, zoom, and jitter are sampled independently per image
            # so the model learns invariance to these spatial variations.
            shared_flip = random.random() < 0.5

            a_params = sample_augment_params(shared_flip=shared_flip)
            p_params = sample_augment_params(shared_flip=shared_flip)
            n_params = sample_augment_params(shared_flip=shared_flip)

            anchor   = self._load_augmented(a_path, a_params)
            positive = self._load_augmented(p_path, p_params)
            negative = self._load_augmented(n_path, n_params)
        else:
            anchor   = self._load_infer(a_path)
            positive = self._load_infer(p_path)
            negative = self._load_infer(n_path)

        return anchor, positive, negative, torch.tensor([1], dtype=torch.float32)

    def _load_augmented(self, path, augment_params):
        """Load one image and apply pre-sampled augmentation params."""
        img = Image.open(path).convert('RGB')
        return preprocess_image(
            img,
            img_size=self.input_shape,
            augment=False,
            augment_params=augment_params
        )

    def _load_infer(self, path):
        """Load one image without augmentation (val/test)."""
        img = Image.open(path).convert('RGB')
        if self.val_transform:
            return self.val_transform(img)
        return preprocess_image(img, img_size=self.input_shape, augment=False)
 
 
class SplitPairDataset(Dataset):
    """
    Pairwise dataset for validation and test evaluation.
 
    Builds (support, query, label) pairs where:
        label = 1  →  both images are genuine (same writer)
        label = 0  →  support is genuine, query is a skilled forgery
 
    For each writer:
        - Genuine pairs  : all C(n_genuine, 2) combinations
        - Forgery pairs  : genuine[i] paired with each forged sample
 
    Args:
        user_dict    (dict)     : {uid: {'genuine': [...paths], 'forged': [...paths]}}
        input_shape  (tuple)    : Canvas size for preprocess_image. Default (224, 224).
        transform    (callable) : Validation transform (no augmentation).
    """
 
    def __init__(self, user_dict, input_shape=(224, 224), transform=None):
        self.input_shape = input_shape
        self.transform   = transform
        self.pairs       = []
 
        for uid, data in user_dict.items():
            gen_key  = next((k for k in data if k.lower() in ('genuine', 'gen')),              None)
            forg_key = next((k for k in data if k.lower() in ('forged', 'forgeries', 'forg')), None)
            gen_paths  = data.get(gen_key,  []) if gen_key  else []
            forg_paths = data.get(forg_key, []) if forg_key else []
 
            # Genuine–Genuine pairs (label = 1)
            for i in range(len(gen_paths)):
                for j in range(i + 1, len(gen_paths)):
                    self.pairs.append((gen_paths[i], gen_paths[j], 1))
 
            # Genuine–Forged pairs (label = 0)
            for g_path in gen_paths:
                for f_path in forg_paths:
                    self.pairs.append((g_path, f_path, 0))
 
        print(f"   PairDataset: {len(self.pairs)} pairs "
              f"({sum(1 for _,_,l in self.pairs if l==1)} genuine, "
              f"{sum(1 for _,_,l in self.pairs if l==0)} forged)")
 
    def __len__(self):
        return len(self.pairs)
 
    def __getitem__(self, idx):
        sup_path, qry_path, label = self.pairs[idx]
        sup_img = self._load(sup_path)
        qry_img = self._load(qry_path)
        return sup_img, qry_img, torch.tensor(label, dtype=torch.float32)
 
    def _load(self, path):
        img = Image.open(path).convert('RGB')
        if self.transform:
            return self.transform(img)
        return preprocess_image(img, img_size=self.input_shape, augment=False)
 
 
print(" > [Dataset] SplitTripletDataset defined")
print(" > [Dataset] SplitPairDataset defined")

 > [Dataset] SplitTripletDataset defined
 > [Dataset] SplitPairDataset defined


## Step 5 — Training Utilities

In [ ]:
def freeze_backbone(fe):
    """Freeze all DenseNet-121 backbone parameters (Phase 1)."""
    for p in fe.get_backbone_params():
        p.requires_grad = False


def unfreeze_backbone(fe):
    """Unfreeze all model parameters for full fine-tuning (Phase 2)."""
    for p in fe.parameters():
        p.requires_grad = True


def validate(fe, loader, device):
    """
    Evaluate using pairwise SED on the unit hypersphere.
    Identical to proposed model validate(). See that notebook for full docstring.
    """
    fe.eval()
    all_scores, all_labels = [], []

    with torch.no_grad():
        for sup_imgs, qry_imgs, labels in loader:
            sup_imgs = sup_imgs.to(device, non_blocking=True)
            qry_imgs = qry_imgs.to(device, non_blocking=True)

            sup_feat  = fe(sup_imgs)
            qry_feat  = fe(qry_imgs)
            distances = torch.sum((sup_feat - qry_feat) ** 2, dim=1)
            scores    = 1.0 - (distances / 4.0)

            all_scores.extend(scores.cpu().numpy().tolist())
            all_labels.extend(labels.numpy().tolist())

    return compute_metrics(all_labels, all_scores)


def evaluate_model(fe, loader, device, output_dir=None, silent=False):
    metrics = validate(fe, loader, device)

    if output_dir and not silent:
        print(f"\n{'='*10} FINAL TEST RESULTS {'='*10}")
        for k, fmt in [('eer',       ':.2%'),
                       ('auc',       ':.4f'),
                       ('threshold', ':.4f'),
                       ('accuracy',  ':.2%'),
                       ('precision', ':.2%'),
                       ('recall',    ':.2%'),
                       ('f1',        ':.2%')]:
            print(f"  {k.upper():<13}: {metrics.get(k, 0):{fmt[1:]}}")
        print("=" * 38)
        _plot_roc_curve(metrics, output_dir)
        _plot_score_distribution(metrics, output_dir)
        _plot_confusion_matrix(metrics, output_dir)
        _plot_det_curve(metrics, output_dir)
        _plot_far_frr(metrics, output_dir)

    return metrics


def run_training(train_user_dict, val_user_dict, test_user_dict,
                 device, checkpoint_path,
                 epochs, phase1_epochs, lr,
                 margin, weight_decay, batch_size,
                 output_dir=None):
    """
    Ablation B training loop — Triplet metric learning, no CBAM.

    Training protocol is identical to the proposed model in every respect.
    The only difference is the feature extractor instantiation:
        baseline=True  → no CBAM modules
        normalize=True → L2 normalization applied (required for SED metric)

    This isolates the contribution of the Triplet Network alone.
    """
    VAL_EVERY = 3

    print(f"\n   {'─'*60}")
    print(f"   ABLATION B — DenseNet-121 + Triplet Network  |  No CBAM")
    print(f"   Epochs: {epochs} (P1 frozen: {phase1_epochs})")
    print(f"   LR: {lr}  |  Margin: {margin}  |  "
          f"WD: {weight_decay}  |  Batch: {batch_size}")
    print(f"   CBAM: OFF  |  L2 Norm: ON  |  Loss: TripletLoss (SED)")
    print(f"   {'─'*60}")

    seed_everything(42)
    t0 = time.time()

    train_dataset = SplitTripletDataset(
        train_user_dict, input_shape=INPUT_SHAPE,
        val_transform=val_transform, training=True
    )
    val_dataset  = SplitPairDataset(val_user_dict,
                                     input_shape=INPUT_SHAPE,
                                     transform=val_transform)
    test_dataset = SplitPairDataset(test_user_dict,
                                     input_shape=INPUT_SHAPE,
                                     transform=val_transform)

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=(NUM_WORKERS > 0)
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
        persistent_workers=(NUM_WORKERS > 0)
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=False,
        persistent_workers=(NUM_WORKERS > 0)
    )

    # ── Model ─────────────────────────────────────────────────────────────────
    # baseline=True   : no CBAM — this is the ablation variable
    # normalize=True  : L2 normalization ON — required for SED metric learning
    # output_dim=1024 : metric embedding, not classification logits
    model = DenseNetFeatureExtractor(
        backbone_name='densenet121',
        output_dim=FEATURE_DIM,
        pretrained=True,
        baseline=True,
        normalize=True
    ).to(device)

    criterion = TripletLoss(margin=margin, mode='euclidean')
    scaler    = torch.amp.GradScaler('cuda')

    freeze_backbone(model)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"   Phase 1: {n_train:,} / {n_total:,} params trainable "
          f"(backbone frozen)")

    optimizer    = optim.AdamW(model.get_head_params(),
                               lr=lr, weight_decay=weight_decay)
    scheduler    = None
    best_eer     = float('inf')
    best_metrics = {}
    history      = {
        'train_loss':      [],
        'active_fraction': [],
        'val_eer':         [],
        'val_epochs':      []
    }

    for epoch in range(epochs):

        if epoch == phase1_epochs:
            unfreeze_backbone(model)
            n_train = sum(p.numel() for p in model.parameters()
                          if p.requires_grad)
            print(f"   Phase 2: {n_train:,} params trainable "
                  f"(backbone unfrozen)")
            optimizer = optim.AdamW([
                {'params': model.get_backbone_params(), 'lr': lr * 0.1},
                {'params': model.get_head_params(),     'lr': lr}
            ], weight_decay=weight_decay)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6
            )

        model.train()
        epoch_loss = 0.0

        for anchor, pos, neg, _ in tqdm(
                train_loader, desc=f"Train E{epoch+1:02d}", leave=False):
            anchor = anchor.to(device, non_blocking=True)
            pos    = pos.to(device,    non_blocking=True)
            neg    = neg.to(device,    non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda'):
                a_emb = model(anchor)
                p_emb = model(pos)
                n_emb = model(neg)
                loss  = criterion(a_emb, p_emb, n_emb)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        phase    = 1 if epoch < phase1_epochs else 2
        history['train_loss'].append(avg_loss)
        history['active_fraction'].append(criterion.last_fraction_active)

        should_validate = (
            (epoch + 1) % VAL_EVERY == 0 or
            (epoch + 1) == epochs
        )

        if should_validate:
            val_metrics = validate(model, val_loader, device)
            val_eer     = val_metrics['eer']
            val_acc     = val_metrics['accuracy']
            history['val_eer'].append(val_eer)
            history['val_epochs'].append(epoch + 1)

            print(f"   [P{phase}] Epoch {epoch+1:02d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Active: {criterion.last_fraction_active:.1%} | "
                  f"Val EER: {val_eer:.2%} | "
                  f"Val Acc: {val_acc:.2%}")

            if scheduler is not None:
                scheduler.step(val_eer)

            if val_eer < best_eer:
                best_eer     = val_eer
                best_metrics = val_metrics
                torch.save({
                    'feature_extractor': model.state_dict(),
                    'metrics': {
                        k: float(v) for k, v in best_metrics.items()
                        if isinstance(v, (int, float,
                                          np.floating, np.integer))
                    }
                }, checkpoint_path)
                print(f"   >>> Checkpoint saved  (Val EER: {val_eer:.2%})")
        else:
            print(f"   [P{phase}] Epoch {epoch+1:02d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | "
                  f"Active: {criterion.last_fraction_active:.1%} | "
                  f"(skipping val)")

        train_dataset._generate_triplets()

    # ── Final test evaluation ─────────────────────────────────────────────────
    if os.path.exists(checkpoint_path):
        ckpt = torch.load(checkpoint_path, map_location=device,
                          weights_only=False)
        state = (ckpt['feature_extractor']
                 if 'feature_extractor' in ckpt else ckpt)
        model.load_state_dict(state, strict=True)
        print("   Best checkpoint reloaded for final test evaluation")

    final_metrics = evaluate_model(
        model, test_loader, device,
        output_dir=output_dir, silent=False
    )

    print(f"\n   Training done | Best Val EER: {best_eer:.2%} | "
          f"Time: {time.time() - t0:.1f}s")

    return final_metrics, history


print(" > [Utils] All training functions defined")


 > [Utils] All training functions defined


## Step 6 — Run All Splits

In [ ]:
all_results = {}
all_histories = {}

for ratio in SPLIT_RATIOS:
    split_file = os.path.join(SPLIT_DIR, f"{DATASET}_split_{ratio}.json")
    tr_p, va_p, te_p = ratio.split('_')
    split_label    = f"{tr_p}:{va_p}:{te_p}"
    split_eval_dir = os.path.join(EVAL_DIR,
                                  f"{NOTEBOOK_NAME}_{tr_p}-{va_p}-{te_p}")
    exp_dir        = os.path.join(CHECKPOINT_DIR, f"{DATASET}_{ratio}_ablB")
    os.makedirs(split_eval_dir, exist_ok=True)
    os.makedirs(exp_dir,        exist_ok=True)

    print(f"\n{'='*70}")
    print(f"  Ablation B — {DATASET_NAME}  |  {split_label}  Split")
    print(f"  DenseNet-121 + Triplet Network  |  No CBAM")
    print(f"{'='*70}")

    if not os.path.exists(split_file):
        print(f"  SKIPPED: split file not found ({split_file})")
        continue

    with open(split_file) as f:
        split_data = json.load(f)

    train_dict = split_data['train']
    val_dict   = split_data['val']
    test_dict  = split_data['test']

    train_writers = set(train_dict.keys())
    val_writers   = set(val_dict.keys())
    test_writers  = set(test_dict.keys())
    assert len(train_writers & val_writers)  == 0, "DATA LEAK: train/val"
    assert len(train_writers & test_writers) == 0, "DATA LEAK: train/test"
    assert len(val_writers   & test_writers) == 0, "DATA LEAK: val/test"

    print(f"  Writers — Train: {len(train_dict)} | "
          f"Val: {len(val_dict)} | Test: {len(test_dict)}")
    print(f"  Split integrity: VERIFIED")

    seed_everything(42)
    checkpoint_path = os.path.join(exp_dir, "best_ablB_model.pth")
    t0_train = time.time()

    best_metrics, history = run_training(
        train_user_dict=train_dict,
        val_user_dict=val_dict,
        test_user_dict=test_dict,
        device=DEVICE,
        checkpoint_path=checkpoint_path,
        epochs=TRAIN_EPOCHS,
        phase1_epochs=TRAIN_PHASE1_EPOCHS,
        lr=TRAIN_LR,
        margin=TRAIN_MARGIN,
        weight_decay=TRAIN_WEIGHT_DECAY,
        batch_size=TRAIN_BATCH_SIZE,
        output_dir=split_eval_dir
    )
    t_train = time.time() - t0_train
    all_histories[split_label] = history

    # ── Training history plot ─────────────────────────────────────────────────
    n_epochs_run = len(history['train_loss'])
    epoch_axis   = list(range(1, n_epochs_run + 1))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    fig.suptitle(
        f'Ablation B — DenseNet-121 + Triplet | {DATASET_NAME} ({split_label})',
        fontsize=13, fontweight='bold'
    )
    ax1.plot(epoch_axis, history['train_loss'],
             color='steelblue', linewidth=2, label='Train Loss')
    ax1.axvline(x=TRAIN_PHASE1_EPOCHS, color='gray', linestyle='--',
                linewidth=1.2, label=f'Phase transition (ep {TRAIN_PHASE1_EPOCHS})')
    ax1.set_ylabel('Triplet Loss', fontsize=11)
    ax1.legend(fontsize=10)
    ax1.grid(True, linestyle='--', alpha=0.4)

    ax2.plot(epoch_axis, [f * 100 for f in history['active_fraction']],
             color='firebrick', linewidth=2, label='Active triplets (%)')
    ax2.axvline(x=TRAIN_PHASE1_EPOCHS, color='gray', linestyle='--',
                linewidth=1.2)
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Active Triplets (%)', fontsize=11)
    ax2.set_ylim(0, 105)
    ax2.legend(fontsize=10)
    ax2.grid(True, linestyle='--', alpha=0.4)

    for ve in history['val_epochs']:
        ax1.axvline(x=ve, color='seagreen', linestyle=':', alpha=0.3)
        ax2.axvline(x=ve, color='seagreen', linestyle=':', alpha=0.3)

    plt.tight_layout()
    history_path = os.path.join(split_eval_dir, 'training_history.png')
    plt.savefig(history_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f" > Training history saved → {history_path}")

    key = f"{DATASET_NAME} ({split_label})"
    all_results[key] = {
        'dataset':            DATASET_NAME,
        'split':              split_label,
        'ablation':           'B — Triplet only (no CBAM)',
        'train_users':        len(train_dict),
        'val_users':          len(val_dict),
        'test_users':         len(test_dict),
        'eer':                float(best_metrics['eer']),
        'accuracy':           float(best_metrics['accuracy']),
        'auc':                float(best_metrics['auc']),
        'precision':          float(best_metrics.get('precision', 0)),
        'recall':             float(best_metrics.get('recall',    0)),
        'f1':                 float(best_metrics.get('f1',        0)),
        'train_time_seconds': round(t_train, 2),
    }

    print(f"  RESULT [{split_label}]: "
          f"EER={best_metrics['eer']:.2%} | "
          f"Acc={best_metrics['accuracy']:.2%} | "
          f"AUC={best_metrics['auc']:.4f} | "
          f"F1={best_metrics.get('f1', 0):.4f} | "
          f"Time={t_train:.1f}s")

print(f"\n{'='*70}\nALL EXPERIMENTS COMPLETE\n{'='*70}")


  BHSig-Bengali  —  70:15:15  Split
 > [Config] Dataset: BHSig-Bengali | Split: 70:15:15
 > [Config] Triplet Training: 100 ep (P1 frozen: 20) | margin=1.0
 > [Config] LR: 0.0001 | WD: 0.0001 | Checkpoints: /home/lawrence/workspace/thesis/thesis/checkpoints/proposed_splits
  Writers — Train: 70 | Val: 15 | Test: 15
 > [Seed] 42

   ────────────────────────────────────────────────────────────
   TRIPLET TRAINING  |  100 epochs  (P1 frozen: 20)
   LR: 0.0001  |  Margin: 1.0  |  WD: 0.0001  |  Batch: 32
   ────────────────────────────────────────────────────────────
 > [Seed] 42
   TripletDataset: 1680 triplets | 70 users | triplet-level aug
   PairDataset: 14940 pairs (4140 genuine, 10800 forged)
   PairDataset: 14940 pairs (4140 genuine, 10800 forged)
   Phase 1: 1,658,248 / 8,612,104 params trainable (backbone frozen)


Train E01:   0%|          | 0/52 [00:00<?, ?it/s]

   [P1] Epoch 01/100 | Loss: 0.7562 | Active: 96.9% | (skipping val)


Train E02:   0%|          | 0/52 [00:00<?, ?it/s]

   [P1] Epoch 02/100 | Loss: 0.6614 | Active: 75.0% | (skipping val)


...

   [P2] Epoch 98/100 | Loss: 0.0772 | Active: 0.0% | (skipping val)


Train E99:   0%|          | 0/52 [00:00<?, ?it/s]

   [P2] Epoch 99/100 | Loss: 0.0989 | Active: 0.0% | Val EER: 15.60% | Val Acc: 84.40%


Train E100:   0%|          | 0/52 [00:00<?, ?it/s]

   [P2] Epoch 100/100 | Loss: 0.0798 | Active: 0.0% | Val EER: 15.28% | Val Acc: 84.72%
   Best checkpoint reloaded for final test evaluation

========== FINAL TEST RESULTS ==========
  EER          : 11.29%
  AUC          : 0.9621
  THRESHOLD    : 0.7790
  ACCURACY     : 88.71%
  PRECISION    : 75.08%
  RECALL       : 88.70%
  F1           : 81.32%
 > Saved ROC Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/tDCBAM_bengali/tDCBAM_bengali_70-15-15/roc_curve.png
 > Saved Distribution Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/tDCBAM_bengali/tDCBAM_bengali_70-15-15/score_distribution.png
 > Saved Confusion Matrix to: /home/lawrence/workspace/thesis/thesis/model_evals/tDCBAM_bengali/tDCBAM_bengali_70-15-15/confusion_matrix.png
 > Saved DET Curve to: /home/lawrence/workspace/thesis/thesis/model_evals/tDCBAM_bengali/tDCBAM_bengali_70-15-15/det_curve.png
 > Saved FAR/FRR Plot to: /home/lawrence/workspace/thesis/thesis/model_evals/tDCBAM_bengali/tDCBAM_bengali_70-

## Step 7 — Summary Table + Bar Chart

In [ ]:
W = 100
print(f"\n{'='*W}")
print(f"{'ABLATION B — DenseNet-121 + Triplet Network (No CBAM)':^{W}}")
print(f"{'='*W}")
print(f"{'Split':<10} {'Train':<8} {'Val':<8} {'Test':<8} "
      f"{'EER':>8} {'Accuracy':>10} {'AUC':>8} "
      f"{'F1':>8} {'Time(s)':>10}")
print(f"{'-'*W}")

for key, res in all_results.items():
    print(f"{res['split']:<10} {res['train_users']:<8} "
          f"{res['val_users']:<8} {res['test_users']:<8} "
          f"{res['eer']:>8.4f} {res['accuracy']:>10.4f} "
          f"{res['auc']:>8.4f} {res['f1']:>8.4f} "
          f"{res['train_time_seconds']:>10.2f}")

print(f"{'='*W}")

results_path = os.path.join(CHECKPOINT_DIR,
                             f'ablation_B_{DATASET}_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\n > Results saved → {results_path}")

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle(
    f'Ablation B — DenseNet-121 + Triplet Network (No CBAM) | {DATASET_NAME}',
    fontsize=13, fontweight='bold'
)
for ax, metric, title, color in zip(
        axes,
        ['eer',        'accuracy',  'auc',   'f1'],
        ['EER (↓)',    'Accuracy',  'AUC',   'F1-Score'],
        ['steelblue',  'seagreen',  'coral', 'mediumpurple']):
    labels = [r.replace('_', ':') for r in SPLIT_RATIOS]
    values = [all_results.get(
        f"{DATASET_NAME} ({r.replace('_',':')})", {}
    ).get(metric, 0) for r in SPLIT_RATIOS]
    bars = ax.bar(labels, values, color=color, alpha=0.8, edgecolor='black')
    ax.set_xlabel('Split Ratio', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.004,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plot_path = os.path.join(CHECKPOINT_DIR, f'ablation_B_{DATASET}_comparison.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" > Plot saved → {plot_path}")


                          PROPOSED MODEL  —  BHSig-Bengali                          
                         DenseNet-121 + CBAM + Triplet Loss                         
Split         Train    Val   Test       EER  Accuracy      AUC       F1   Time(s)
------------------------------------------------------------------------------------
70:15:15         70     15     15    0.1129    0.8871   0.9621   0.8132    1536.8

 > Results saved → /home/lawrence/workspace/thesis/thesis/checkpoints/proposed_splits/proposed_bhsig_bengali_results.json
